# ONNX Runtime MEMO Example

This notebook shows how to run a single image with the MEMO ONNX Runtime deployment path and display the results inline.

The example image is downloaded automatically from a public URL, so this notebook does not require a local edge_data folder.

## What This Notebook Does

- Loads the split ONNX MEMO predictor
- Lists the downloadable public example images
- Downloads one example image into experiments/demo_examples
- Runs ONNX Runtime inference
- Displays the original image, prediction map, and binarized result

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
from PIL import Image
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'deployment_onnx' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo_example_assets import ensure_demo_example, list_demo_examples
from deployment_onnx.onnx_runtime import ONNXMEMOPredictor
from deployment_onnx.runtime_selector import recommend_runtime
from onnx_model_registry import list_onnx_model_presets, resolve_onnx_model_preset

In [ ]:
runtime_info = recommend_runtime()
available_models = {name: preset['description'] for name, preset in list_onnx_model_presets().items()}
available_examples = {name: metadata['description'] for name, metadata in list_demo_examples().items()}
{'runtime_info': runtime_info, 'models': available_models, 'examples': available_examples}

In [ ]:
config = {
    'model_name': 'BIPED Tiny INT8',
    'example_name': 'Lena',
    'guidance_scale': 1.8,
    'max_steps': 20,
    'resize_long_side': 320,
    'runtime_variant': None,
}

preset = resolve_onnx_model_preset(config['model_name'])
config['runtime_variant'] = config['runtime_variant'] or preset.get('runtime_variant', 'auto')
config['dino_encoder_path'] = Path(preset['dino_encoder_path'])
config['denoiser_path'] = Path(preset['denoiser_path'])
config['image_path'] = ensure_demo_example(config['example_name'])
config

In [ ]:
predictor = ONNXMEMOPredictor(
    dino_encoder_path=str(config['dino_encoder_path']),
    denoiser_path=str(config['denoiser_path']),
    guidance_scale=config['guidance_scale'],
    max_steps=config['max_steps'],
    resize_long_side=config['resize_long_side'],
    runtime_variant=config['runtime_variant'],
)

print('model:', config['model_name'])
print('dino_encoder_path:', config['dino_encoder_path'])
print('denoiser_path:', config['denoiser_path'])

In [ ]:
image_bgr = cv2.imread(str(config['image_path']), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise FileNotFoundError(config['image_path'])

result = predictor.predict_bgr(image_bgr)

original_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
prediction_rgb = cv2.cvtColor(result['prediction'], cv2.COLOR_GRAY2RGB)
binarized_rgb = cv2.cvtColor(result['binarized'], cv2.COLOR_GRAY2RGB)

print('model:', config['model_name'])
print('original shape:', original_rgb.shape)
print('prediction shape:', result['prediction'].shape)
print('binarized shape:', result['binarized'].shape)
print('prediction mean:', float(result['prediction'].mean()))
print('binarized foreground ratio:', float((result['binarized'] > 0).mean()))

In [ ]:
def add_title_bar(image_rgb: np.ndarray, title: str, bar_height: int = 28) -> np.ndarray:
    canvas = np.full((image_rgb.shape[0] + bar_height, image_rgb.shape[1], 3), 255, dtype=np.uint8)
    canvas[bar_height:] = image_rgb
    cv2.putText(
        canvas,
        title,
        (8, 19),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (20, 20, 20),
        1,
        cv2.LINE_AA,
    )
    return canvas

panels = [
    add_title_bar(original_rgb, 'Original'),
    add_title_bar(prediction_rgb, 'Prediction'),
    add_title_bar(binarized_rgb, 'Binarized'),
]

max_height = max(panel.shape[0] for panel in panels)
normalized_panels = []
for panel in panels:
    if panel.shape[0] < max_height:
        pad = np.full((max_height - panel.shape[0], panel.shape[1], 3), 255, dtype=np.uint8)
        panel = np.concatenate([panel, pad], axis=0)
    normalized_panels.append(panel)

preview = np.concatenate(normalized_panels, axis=1)
display(Image.fromarray(preview))

In [ ]:
save_dir = PROJECT_ROOT / 'experiments' / 'onnx_notebook_demo_tiny'
save_dir.mkdir(parents=True, exist_ok=True)

cv2.imwrite(str(save_dir / 'example_prediction.png'), result['prediction'])
cv2.imwrite(str(save_dir / 'example_binarized.png'), result['binarized'])

print('saved to:', save_dir)

In [ ]:
from demo_model_registry import resolve_model_preset
from deployment.memo_runtime import OptimizedMEMOPredictor

pytorch_preset = resolve_model_preset('BIPED Tiny')
pytorch_predictor = OptimizedMEMOPredictor(
    config_file=pytorch_preset['config_file'],
    model_path=pytorch_preset['model_path'],
    base_model_path=pytorch_preset['base_model_path'],
    device='auto',
    precision='fp16',
)

pytorch_result = pytorch_predictor.predict_bgr(
    image_bgr,
    guidance_scale=config['guidance_scale'],
    max_steps=config['max_steps'],
    resize_long_side=config['resize_long_side'],
)

pixel_agreement = float((pytorch_result['binarized'] == result['binarized']).mean())
prediction_mae = float(np.abs(pytorch_result['prediction'].astype(np.float32) - result['prediction'].astype(np.float32)).mean())
{'binary_agreement': pixel_agreement, 'prediction_mae': prediction_mae}